# Launch outcomes and data preparation

This notebook derives the landing class from the recorded outcome and verifies every row against the supplied reference. The IBM snapshot contains 90 launches; its observed outcomes are not a forecast.

Run the cells in order from the repository root or `notebooks/`. Install `requirements.txt` and run `download_inputs.py` if `data/` is missing. Local output files are written under `results/`.

## Sources and credit

Adapted from this repository's `scripts/01_data_wrangling.py` and the public IBM DS0321EN Skills Network SpaceX labs. IBM and the credited lab authors, including Joseph Santarcangelo, Nayef Abou Tayoun and Yan Luo, retain their source credit. Input URLs and SHA-256 hashes are listed in `SOURCES.json`; see `ATTRIBUTION.md` for third-party terms. This notebook does not grant a new licence over third-party material.


In [1]:
import warnings
warnings.filterwarnings('ignore', category=FutureWarning)
import pandas as pd
pd.set_option('display.max_rows', 12)
pd.set_option('display.max_columns', 12)
pd.set_option('display.max_colwidth', 70)

In [2]:
from pathlib import Path

import os

import matplotlib

matplotlib.use('module://matplotlib_inline.backend_inline')

from IPython.display import display

HERE = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'data').is_dir() and (p / 'SOURCES.json').is_file()), None)

if HERE is None:
    raise FileNotFoundError('Open this notebook from the repository root or its notebooks folder; run download_inputs.py first.')

DATA = HERE / 'data'

OUT = HERE / 'results' / '01_wrangling'

OUT.mkdir(parents=True, exist_ok=True)

os.chdir(OUT)

import pandas as pd

import numpy as np

df = pd.read_csv(DATA / 'dataset_part_1.csv')

df.head(10)

df.isnull().sum() / len(df) * 100

df.dtypes

launch_site_counts = df['LaunchSite'].value_counts()

launch_site_counts

LaunchSite
CCAFS SLC 40    55
KSC LC 39A      22
VAFB SLC 4E     13
Name: count, dtype: int64

In [3]:
orbit_counts = df['Orbit'].value_counts()

orbit_counts

landing_outcomes = df['Outcome'].value_counts()

landing_outcomes

for i, outcome in enumerate(landing_outcomes.keys()):
    print(i, outcome)

bad_outcomes = set(landing_outcomes.keys()[[1, 3, 5, 6, 7]])

bad_outcomes

landing_class = [0 if outcome in bad_outcomes else 1 for outcome in df['Outcome']]

df['Class'] = landing_class

df[['Class']].head(8)

df.head(5)

df['Class'].mean()

import json

assert len(df) == 90

assert len(df['Class']) == len(df)

reference = pd.read_csv(DATA / 'dataset_part_2.csv')

assert df['FlightNumber'].equals(reference['FlightNumber'])

assert df['Class'].equals(reference['Class'])

assert df['Class'].equals(df['Outcome'].str.startswith('True').astype(int))

0 True ASDS
1 None None
2 True RTLS
3 False ASDS
4 True Ocean
5 False Ocean
6 None ASDS
7 False RTLS


In [4]:
df.to_csv(OUT / 'dataset_part_2.derived.csv', index=False)

result = {'rows': len(df), 'successes': int(df['Class'].sum()), 'success_rate': float(df['Class'].mean()), 'official_Class_column_matches_all_rows': True, 'site_counts': launch_site_counts.to_dict(), 'bad_outcomes': sorted(bad_outcomes)}

(OUT / 'RESULTS.json').write_text(json.dumps(result, indent=2))

print(json.dumps(result, indent=2))

{
  "rows": 90,
  "successes": 60,
  "success_rate": 0.6666666666666666,
  "official_Class_column_matches_all_rows": true,
  "site_counts": {
    "CCAFS SLC 40": 55,
    "KSC LC 39A": 22,
    "VAFB SLC 4E": 13
  },
  "bad_outcomes": [
    "False ASDS",
    "False Ocean",
    "False RTLS",
    "None ASDS",
    "None None"
  ]
}
